In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, roc_auc_score

from imblearn.over_sampling import SMOTE

import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
import pandas as pd

df = pd.read_csv("creditcard1.csv")
print (df.shape)
df.head()

(284807, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


**PREPROCESSING**

In [3]:
X = df.drop("Class", axis=1)
y = df["Class"]

In [4]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

In [5]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
smote = SMOTE(random_state=42)
auc_scores = []

In [6]:
for train_index, test_index in skf.split(X_scaled, y):

    X_train, X_test = X_scaled.iloc[train_index], X_scaled.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # SMOTE hanya untuk training
    X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

    input_dim = X_train_res.shape[1]

In [7]:
input_layer = Input(shape=(input_dim,))

# Encoder
encoded = Dense(512, activation="relu", kernel_regularizer=l2(0.0001))(input_layer)
encoded = BatchNormalization()(encoded)
encoded = Dropout(0.2)(encoded)

encoded = Dense(256, activation="relu", kernel_regularizer=l2(0.0001))(encoded)
encoded = BatchNormalization()(encoded)
encoded = Dropout(0.2)(encoded)

encoded = Dense(128, activation="relu", kernel_regularizer=l2(0.0001))(encoded)
encoded = BatchNormalization()(encoded)

encoded = Dense(64, activation="relu", kernel_regularizer=l2(0.0001))(encoded)
encoded = BatchNormalization()(encoded)

# Decoder
decoded = Dense(128, activation="relu")(encoded)
decoded = BatchNormalization()(decoded)

decoded = Dense(256, activation="relu")(decoded)
decoded = BatchNormalization()(decoded)

decoded = Dense(512, activation="relu")(decoded)
decoded = BatchNormalization()(decoded)

decoded = Dense(input_dim, activation="linear")(decoded)

autoencoder = Model(inputs=input_layer, outputs=decoded)

In [8]:
autoencoder.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
        loss="mse"
)
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [9]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6
)

In [10]:
history = autoencoder.fit(
    X_train_res,
    X_train_res,
    epochs=50,
    batch_size=64,
    validation_split=0.2,
    shuffle=True,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/50
5687/5687 ━━━━━━━━━━━━━━━━━━━━ 49s 7ms/step - loss: 1.0110 - val_loss: 0.3803 - learning_rate: 1.0000e-04
Epoch 2/50
5687/5687 ━━━━━━━━━━━━━━━━━━━━ 41s 7ms/step - loss: 0.4117 - val_loss: 0.1967 - learning_rate: 1.0000e-04
Epoch 3/50
5687/5687 ━━━━━━━━━━━━━━━━━━━━ 40s 7ms/step - loss: 0.3357 - val_loss: 0.1329 - learning_rate: 1.0000e-04
Epoch 4/50
5687/5687 ━━━━━━━━━━━━━━━━━━━━ 38s 7ms/step - loss: 0.2956 - val_loss: 0.1059 - learning_rate: 1.0000e-04
Epoch 5/50
5687/5687 ━━━━━━━━━━━━━━━━━━━━ 39s 7ms/step - loss: 0.2719 - val_loss: 0.1211 - learning_rate: 1.0000e-04
Epoch 6/50
5687/5687 ━━━━━━━━━━━━━━━━━━━━ 41s 7ms/step - loss: 0.2517 - val_loss: 0.0776 - learning_rate: 1.0000e-04
Epoch 7/50
5687/5687 ━━━━━━━━━━━━━━━━━━━━ 88s 8ms/step - loss: 0.2394 - val_loss: 0.0700 - learning_rate: 1.0000e-04
Epoch 8/50
5687/5687 ━━━━━━━━━━━━━━━━━━━━ 44s 8ms/step - loss: 0.2276 - val_loss: 0.0689 - learning_rate: 1.0000e-04
Epoch 9/50
5687/5687 ━━━━━━━━━━━━━━━━━━━━ 41s 7ms/step - loss: 0

In [18]:
reconstructions = autoencoder.predict(X_test)
mse = np.mean(np.power(X_test - reconstructions, 2), axis=1)

auc = roc_auc_score(y_test, mse)



1781/1781 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step


In [19]:
from sklearn.metrics import roc_curve

fpr, tpr, thresholds = roc_curve(y_test, mse)

optimal_idx = np.argmax(tpr - fpr)

threshold = thresholds[optimal_idx]

y_pred = (mse > threshold).astype(int)

In [21]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      0.94      0.97     56863
           1       0.02      0.76      0.04        98

    accuracy                           0.94     56961
   macro avg       0.51      0.85      0.51     56961
weighted avg       1.00      0.94      0.97     56961



In [20]:
print("AUC:", auc)

auc_scores.append(auc)

print("Mean AUC:", np.mean(auc_scores))

AUC: 0.9206639517034678
Mean AUC: 0.9206639517034678
